# FL Methods Comparison + Multimodal Exploration

**Goal A**: Compare 6 FL strategies on the same medical skeleton data
**Goal B**: Explore synthetic IMU from 3D skeleton + fusion approaches

## FL Methods Tested

| # | Method | Core Idea | New vs Baseline |
|---|--------|-----------|-----------------|
| 1 | FedAvg | Standard averaging | Baseline (Phase 1) |
| 2 | FedBN | BN kept local | Baseline (Phase 1) |
| 3 | **FedProx** | Proximal term μ/2‖w-w_g‖² | **New** |
| 4 | **Ditto** | Personalized model + L2 reg toward global | **New** |
| 5 | **FedRep** | Shared body + local head | **New** |
| 6 | **APFL** | Adaptive α mixing global/local | **New** |

## Quick-Experiment Settings
- **10 rounds** (not 50) — ~15-20 min per method on T4
- **Same data**: 10 medical classes, 5 clients, Dirichlet α=0.5
- Warm-start: 2 epochs on combined data

In [1]:
# @title Setup {display-mode: "form"}
!pip install -q scikit-learn

import os, time, copy, json, math, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter, defaultdict, OrderedDict
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU"
DEVICE = 'cuda'

PyTorch 2.10.0+cu128  |  CUDA: True
GPU: Tesla T4


In [2]:
# ═══ Quick Experiment Config ═══
SEED = 42
NUM_CLASSES = 10
NUM_CLIENTS = 5
ALPHA = 0.5
BATCH_SIZE = 64
WARMUP_EPOCHS = 2
NUM_ROUNDS = 10          # Quick: 10 rounds (~15 min per method)
EPOCHS_PER_ROUND = 1
LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# FedProx
FEDPROX_MU = 0.01

# Ditto
DITTO_LAMBDA = 0.1

# FedRep
FEDREP_HEAD_EPOCHS = 1
FEDREP_BODY_EPOCHS = 1

# APFL
APFL_ALPHA_INIT = 0.5

MODEL_KWARGS = dict(base_channels=64, num_stages=6, num_person=2, dropout=0.3)

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.makedirs('outputs', exist_ok=True)
print('Quick config set ✓')

Quick config set ✓


## 1. Data Download, Filter & Partition
*(Same pipeline as Phase 1 baselines)*

In [3]:
data_dir = 'data/nturgbd'
pkl_path = f'{data_dir}/ntu60_hrnet.pkl'

if not os.path.exists(pkl_path):
    os.makedirs(data_dir, exist_ok=True)
    !wget -q --show-progress -O {pkl_path} https://download.openmmlab.com/mmaction/pyskl/data/nturgbd/ntu60_hrnet.pkl
    print(f'Downloaded: {pkl_path}')
else:
    print(f'Already exists: {pkl_path}')

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

split_info = data['split']
annotations = data['annotations']

MEDICAL_CLASS_MAP = {
    42: 0, 41: 1, 43: 2, 44: 3, 45: 4, 46: 5, 47: 6, 8: 7, 7: 8, 58: 9,
}
MEDICAL_LABELS = [
    'falling', 'staggering', 'touch head', 'touch chest', 'touch back',
    'touch neck', 'nausea', 'standing up', 'sitting down', 'walking towards',
]
selected = set(MEDICAL_CLASS_MAP.keys())
medical_anns = [a for a in annotations if a['label'] in selected]

def prenormalize_2d(kp, ks, img_shape=(1080, 1920), threshold=0.01):
    kp = kp.astype(np.float32).copy()
    ks = ks.astype(np.float32)
    kp = np.concatenate([kp, ks[..., None]], axis=-1)
    mask = kp[..., 2] <= threshold
    h, w = img_shape
    kp[..., 0] = (kp[..., 0] - w/2) / (w/2)
    kp[..., 1] = (kp[..., 1] - h/2) / (h/2)
    kp[..., 0][mask] = 0; kp[..., 1][mask] = 0
    return kp

def uniform_sample(kp, clip_len=100):
    M, T, V, C = kp.shape
    if T == clip_len: return kp
    inds = np.arange(clip_len) % T if T < clip_len else np.linspace(0, T-1, clip_len, dtype=int)
    return kp[:, inds]

def format_gcn(kp, num_person=2):
    M, T, V, C = kp.shape
    if M < num_person:
        kp = np.concatenate([kp, np.zeros((num_person-M, T, V, C), dtype=kp.dtype)], 0)
    return kp[:num_person]

def preprocess(ann):
    kp = prenormalize_2d(ann['keypoint'], ann['keypoint_score'],
                         ann.get('img_shape', (1080, 1920)))
    return format_gcn(uniform_sample(kp, 100))

train_dirs = set(split_info['xsub_train'])
test_dirs = set(split_info['xsub_val'])

t0 = time.time()
train_x, train_y, test_x, test_y = [], [], [], []
for ann in medical_anns:
    x = preprocess(ann)
    y = MEDICAL_CLASS_MAP[ann['label']]
    if ann['frame_dir'] in train_dirs:
        train_x.append(x); train_y.append(y)
    elif ann['frame_dir'] in test_dirs:
        test_x.append(x); test_y.append(y)

train_x = np.stack(train_x).astype(np.float32)
train_y = np.array(train_y, dtype=np.int64)
test_x = np.stack(test_x).astype(np.float32)
test_y = np.array(test_y, dtype=np.int64)
print(f'Train: {train_x.shape}, Test: {test_x.shape} ({time.time()-t0:.1f}s)')

# Dirichlet split
rng = np.random.RandomState(SEED)
label_indices = defaultdict(list)
for i, label in enumerate(train_y):
    label_indices[int(label)].append(i)

client_indices = {i: [] for i in range(NUM_CLIENTS)}
for cls in range(NUM_CLASSES):
    idxs = np.array(label_indices[cls]); rng.shuffle(idxs)
    props = rng.dirichlet([ALPHA] * NUM_CLIENTS)
    props = np.cumsum(props) / props.sum()
    splits = np.split(idxs, (props[:-1] * len(idxs)).astype(int))
    for cid, chunk in enumerate(splits):
        client_indices[cid].extend(chunk.tolist())

for cid in range(NUM_CLIENTS):
    ci = client_indices[cid]
    print(f'  Client {cid}: {len(ci)} samples, {len(set(train_y[ci]))}/{NUM_CLASSES} classes')

# Build loaders
client_loaders = []
all_x_parts, all_y_parts = [], []
for cid in range(NUM_CLIENTS):
    ci = np.array(client_indices[cid], dtype=int)
    xc, yc = train_x[ci], train_y[ci]
    all_x_parts.append(xc); all_y_parts.append(yc)
    client_loaders.append(DataLoader(
        TensorDataset(torch.from_numpy(xc), torch.from_numpy(yc)),
        batch_size=BATCH_SIZE, shuffle=True))

all_train_x = np.concatenate(all_x_parts)
all_train_y = np.concatenate(all_y_parts)
all_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(all_train_x), torch.from_numpy(all_train_y)),
    batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
calib_loader = DataLoader(
    TensorDataset(torch.from_numpy(all_train_x), torch.from_numpy(all_train_y)),
    batch_size=128, shuffle=True)
test_loader = DataLoader(
    TensorDataset(torch.from_numpy(test_x), torch.from_numpy(test_y)),
    batch_size=128)
print(f'Loaders ready ✓')

data/nturgbd/ntu60_ 100%[===================>] 672.72M  25.9MB/s    in 27s     
Downloaded: data/nturgbd/ntu60_hrnet.pkl
Train: (6695, 2, 100, 17, 3), Test: (2749, 2, 100, 17, 3) (1.5s)
  Client 0: 1798 samples, 9/10 classes
  Client 1: 1078 samples, 10/10 classes
  Client 2: 1648 samples, 10/10 classes
  Client 3: 1253 samples, 10/10 classes
  Client 4: 918 samples, 10/10 classes
Loaders ready ✓


## 2. STGCN++ Model

In [4]:
COCO_INWARD = [
    (15,13),(13,11),(16,14),(14,12),(11,5),(12,6),
    (9,7),(7,5),(10,8),(8,6),(5,0),(6,0),(1,0),(3,1),(2,0),(4,2)]
COCO_NUM_JOINTS = 17; COCO_CENTER = 0

def _build_adj(num_node, edges, center):
    def _norm(mx):
        d = mx.sum(0); d[d==0]=1; return mx/d
    I = np.eye(num_node, dtype=np.float32)
    inw = np.zeros((num_node, num_node), dtype=np.float32)
    outw = np.zeros((num_node, num_node), dtype=np.float32)
    hop = np.full(num_node, 1e9, dtype=int); hop[center] = 0
    adj = {i: [] for i in range(num_node)}
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    q = [center]
    while q:
        cur = q.pop(0)
        for nb in adj[cur]:
            if hop[nb] > hop[cur]+1: hop[nb]=hop[cur]+1; q.append(nb)
    for u, v in edges:
        if hop[u] < hop[v]: inw[v,u] = 1
        else: outw[u,v] = 1
    return np.stack([_norm(I), _norm(inw), _norm(outw)])

def get_coco_adj():
    return _build_adj(COCO_NUM_JOINTS, COCO_INWARD, COCO_CENTER)

class MsTCN(nn.Module):
    def __init__(self, in_ch, out_ch, dilations=(1,2,3,4), stride=1, dropout=0.0):
        super().__init__()
        mid = out_ch // (len(dilations)+2); rem = out_ch - mid*(len(dilations)+2)
        self.branches = nn.ModuleList()
        for d in dilations:
            self.branches.append(nn.Sequential(
                nn.Conv2d(in_ch,mid,1), nn.BatchNorm2d(mid), nn.ReLU(True),
                nn.Conv2d(mid,mid,(3,1),stride=(stride,1),padding=(d,0),dilation=(d,1)),
                nn.BatchNorm2d(mid)))
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch,mid,1), nn.BatchNorm2d(mid), nn.ReLU(True),
            nn.MaxPool2d((3,1),stride=(stride,1),padding=(1,0)), nn.BatchNorm2d(mid)))
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_ch,mid+rem,1), nn.BatchNorm2d(mid+rem)))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        outs = [br(x) for br in self.branches]
        mt = min(o.size(2) for o in outs)
        return self.drop(torch.cat([o[:,:,:mt,:] for o in outs], dim=1))

class UnitGCN(nn.Module):
    def __init__(self, in_ch, out_ch, A, adaptive=True):
        super().__init__()
        K = A.size(0); self.K = K; self.register_buffer('A', A)
        self.conv = nn.ModuleList([nn.Conv2d(in_ch,out_ch,1) for _ in range(K)])
        self.PA = nn.Parameter(A.clone()) if adaptive else None
        self.bn = nn.BatchNorm2d(out_ch)
        self.res = nn.Sequential(nn.Conv2d(in_ch,out_ch,1),nn.BatchNorm2d(out_ch)) if in_ch!=out_ch else nn.Identity()
        for m in self.conv: nn.init.kaiming_normal_(m.weight, mode='fan_out')
    def forward(self, x):
        adj = self.A + self.PA if self.PA is not None else self.A
        out = sum(torch.einsum('nctv,vw->nctw', self.conv[k](x), adj[k]) for k in range(self.K))
        return F.relu(self.bn(out) + self.res(x), inplace=True)

class STGCNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, A, stride=1, dropout=0.0):
        super().__init__()
        self.gcn = UnitGCN(in_ch, out_ch, A)
        self.tcn = MsTCN(out_ch, out_ch, stride=stride, dropout=dropout)
        self.residual = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,1), nn.BatchNorm2d(out_ch),
            nn.AvgPool2d((stride,1)) if stride>1 else nn.Identity()
        ) if in_ch!=out_ch or stride!=1 else nn.Identity()
    def forward(self, x):
        return F.relu(self.tcn(self.gcn(x)) + self.residual(x), inplace=True)

class STGCN(nn.Module):
    def __init__(self, num_classes=10, in_channels=3, base_channels=64,
                 num_stages=6, inflate_stages=None, down_stages=None,
                 num_person=2, dropout=0.0, num_joints=17, adj_fn=get_coco_adj):
        super().__init__()
        if inflate_stages is None: inflate_stages = [3,5]
        if down_stages is None: down_stages = [3,5]
        self.num_person = num_person; self.num_joints = num_joints
        A = torch.tensor(adj_fn(), dtype=torch.float32)
        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)
        channels = [base_channels]
        for i in range(1, num_stages):
            channels.append(channels[-1]*2 if i in inflate_stages else channels[-1])
        self.blocks = nn.ModuleList()
        c_in = in_channels
        for i in range(num_stages):
            self.blocks.append(STGCNBlock(c_in, channels[i], A,
                               stride=2 if i in down_stages else 1, dropout=dropout))
            c_in = channels[i]
        self.out_channels = channels[-1]
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout) if dropout>0 else nn.Identity()
        self.fc = nn.Linear(self.out_channels, num_classes)
        nn.init.normal_(self.fc.weight, 0, math.sqrt(2.0/num_classes))
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        N, M, T, V, C = x.shape
        x = x.permute(0,1,4,2,3).contiguous().view(N*M, C, T, V)
        x = x.permute(0,1,3,2).contiguous().view(N*M, C*V, T)
        x = self.data_bn(x)
        x = x.view(N*M, C, V, T).permute(0,1,3,2).contiguous()
        for block in self.blocks:
            x = block(x)
        x = self.pool(x).view(N, M, self.out_channels).mean(dim=1)
        return self.fc(self.drop(x))

    def get_features(self, x):
        N, M, T, V, C = x.shape
        x = x.permute(0,1,4,2,3).contiguous().view(N*M, C, T, V)
        x = x.permute(0,1,3,2).contiguous().view(N*M, C*V, T)
        x = self.data_bn(x)
        x = x.view(N*M, C, V, T).permute(0,1,3,2).contiguous()
        for block in self.blocks:
            x = block(x)
        return self.pool(x).view(N, M, self.out_channels).mean(dim=1)

def build_model(num_classes=10, in_channels=3, **kw):
    return STGCN(num_classes=num_classes, in_channels=in_channels, **kw)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = build_model(NUM_CLASSES, dropout=0.3)
print(f'STGCN++ params: {count_params(_m):,}')
del _m

STGCN++ params: 443,222


## 3. FL Training Helpers

In [5]:
def _is_bn_param(name):
    return 'bn' in name.lower() or 'data_bn' in name

def calibrate_bn(model, loader, device, max_batches=30):
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(loader):
            if i >= max_batches: break
            model(xb.to(device))

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += yb.size(0)
    return correct / max(total, 1)

def warm_start(model, loader, test_loader, device, epochs=2, lr=0.01):
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()
    for ep in range(1, epochs+1):
        model.train()
        correct, total = 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb); loss = crit(logits, yb)
            loss.backward(); opt.step()
            correct += (logits.argmax(1) == yb).sum().item(); total += yb.size(0)
        test_acc = evaluate(model, test_loader, device)
        print(f'  WS Epoch {ep}/{epochs} train={correct/total:.4f} test={test_acc:.4f}')
    sd = {k: v.cpu() for k, v in model.state_dict().items()}
    model.cpu()
    return sd

print('Helpers defined ✓')

Helpers defined ✓


In [6]:
ws_model = build_model(NUM_CLASSES, in_channels=3, **MODEL_KWARGS)
warmstart_sd = warm_start(ws_model, all_train_loader, test_loader, DEVICE, WARMUP_EPOCHS)
torch.save(warmstart_sd, 'outputs/methods_warmstart.pt')
del ws_model; torch.cuda.empty_cache()
print('Warm-start saved ✓')

  WS Epoch 1/2 train=0.2903 test=0.3834
  WS Epoch 2/2 train=0.4648 test=0.5118
Warm-start saved ✓


## 4. FL Methods Implementation

All methods share the same round structure:
1. Distribute global model
2. Local training (method-specific)
3. Aggregate (method-specific)
4. BN calibration → evaluate

In [7]:
# ═══════════════════════════════════════════════════════════════
# FL Methods: FedAvg, FedBN, FedProx, Ditto, FedRep, APFL
# ═══════════════════════════════════════════════════════════════

def fedavg_aggregate(global_sd, client_weights):
    total = sum(n for n, _ in client_weights)
    return OrderedDict({k: sum(n * sd[k].float() for n, sd in client_weights) / total
                        for k in global_sd})

def fedbn_aggregate(global_sd, client_weights):
    total = sum(n for n, _ in client_weights)
    new_sd = OrderedDict()
    for k in global_sd:
        if _is_bn_param(k):
            new_sd[k] = global_sd[k].clone().float()
        else:
            new_sd[k] = sum(n * sd[k].float() for n, sd in client_weights) / total
    return new_sd

def _local_train(model, loader, device, lr, epochs=1, global_params=None, mu=0.0):
    """Standard local training. If global_params + mu > 0, applies FedProx proximal term."""
    model.to(device).train()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    crit = nn.CrossEntropyLoss()
    correct, total = 0, 0
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            # FedProx proximal term
            if mu > 0 and global_params is not None:
                prox = sum(torch.sum((p - gp.to(device))**2)
                           for p, gp in zip(model.parameters(), global_params))
                loss = loss + (mu / 2) * prox
            loss.backward(); opt.step()
            correct += (logits.argmax(1) == yb).sum().item(); total += yb.size(0)
    return correct / max(total, 1)


def run_fedavg(warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds=NUM_ROUNDS):
    return _run_standard_fl('fedavg', warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds)

def run_fedbn(warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds=NUM_ROUNDS):
    return _run_standard_fl('fedbn', warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds)

def run_fedprox(warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds=NUM_ROUNDS, mu=FEDPROX_MU):
    return _run_standard_fl('fedprox', warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds, mu=mu)


def _run_standard_fl(mode, warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds, mu=0.0):
    global_model = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
    global_model.load_state_dict(copy.deepcopy(warmstart_sd))
    device = torch.device(DEVICE)
    history = []; best_acc = 0.0; best_sd = None
    agg_fn = fedbn_aggregate if mode == 'fedbn' else fedavg_aggregate
    print(f'\n{"="*50}\n{mode.upper()} — {num_rounds} rounds\n{"="*50}')
    t0 = time.time()
    for rnd in range(1, num_rounds+1):
        cur_lr = LR
        global_sd = copy.deepcopy(global_model.state_dict())
        global_params = [p.clone().detach() for p in global_model.parameters()] if mu > 0 else None
        client_weights = []
        for cid, loader in enumerate(client_loaders):
            local = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
            local.load_state_dict(global_sd)
            _local_train(local, loader, device, cur_lr, EPOCHS_PER_ROUND, global_params, mu)
            n = len(loader.dataset)
            client_weights.append((n, {k: v.cpu() for k, v in local.state_dict().items()}))
            del local; torch.cuda.empty_cache()
        new_sd = agg_fn(global_sd, client_weights)
        gsd = global_model.state_dict()
        for k in new_sd: gsd[k] = new_sd[k].to(gsd[k].dtype)
        global_model.load_state_dict(gsd)
        global_model.to(device)
        calibrate_bn(global_model, calib_loader, device)
        test_acc = evaluate(global_model, test_loader, device)
        global_model.cpu()
        if test_acc > best_acc: best_acc = test_acc; best_sd = copy.deepcopy(global_model.state_dict())
        history.append({'round': rnd, 'test_acc': float(test_acc)})
        if rnd % 5 == 0 or rnd == 1:
            print(f'  R{rnd:2d}/{num_rounds} test={test_acc:.4f}')
    print(f'✓ {mode.upper()} done in {time.time()-t0:.0f}s — Best: {best_acc:.4f}')
    return history, best_acc, best_sd


def run_ditto(warmstart_sd, client_loaders, calib_loader, test_loader,
              num_rounds=NUM_ROUNDS, lam=DITTO_LAMBDA):
    """Ditto: global model (FedAvg) + personalized model per client (L2 reg toward global)."""
    global_model = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
    global_model.load_state_dict(copy.deepcopy(warmstart_sd))
    # Each client has a personalized model
    personal_models = [build_model(NUM_CLASSES, 3, **MODEL_KWARGS) for _ in range(NUM_CLIENTS)]
    for pm in personal_models:
        pm.load_state_dict(copy.deepcopy(warmstart_sd))
    device = torch.device(DEVICE)
    history = []; best_acc = 0.0; best_sd = None
    print(f'\n{"="*50}\nDITTO (λ={lam}) — {num_rounds} rounds\n{"="*50}')
    t0 = time.time()
    for rnd in range(1, num_rounds+1):
        global_sd = copy.deepcopy(global_model.state_dict())
        client_weights = []
        personal_accs = []
        for cid, loader in enumerate(client_loaders):
            # Step 1: Train global model copy (standard FedAvg)
            local_g = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
            local_g.load_state_dict(global_sd)
            _local_train(local_g, loader, device, LR, EPOCHS_PER_ROUND)
            n = len(loader.dataset)
            client_weights.append((n, {k: v.cpu() for k, v in local_g.state_dict().items()}))
            del local_g

            # Step 2: Train personalized model with L2 reg toward global
            pm = personal_models[cid]
            pm.to(device).train()
            opt = torch.optim.SGD(pm.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
            crit = nn.CrossEntropyLoss()
            global_params = [p.clone().detach().to(device) for p in global_model.parameters()]
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                logits = pm(xb)
                loss = crit(logits, yb)
                prox = sum(torch.sum((p - gp)**2) for p, gp in zip(pm.parameters(), global_params))
                loss = loss + (lam / 2) * prox
                loss.backward(); opt.step()
            personal_accs.append(evaluate(pm, test_loader, device))
            pm.cpu()
            torch.cuda.empty_cache()

        # Aggregate global model (standard FedAvg)
        new_sd = fedavg_aggregate(global_sd, client_weights)
        gsd = global_model.state_dict()
        for k in new_sd: gsd[k] = new_sd[k].to(gsd[k].dtype)
        global_model.load_state_dict(gsd)

        # Evaluate: mean of personalized model accuracies
        mean_personal = np.mean(personal_accs)
        # Also evaluate global model
        global_model.to(device)
        calibrate_bn(global_model, calib_loader, device)
        global_acc = evaluate(global_model, test_loader, device)
        global_model.cpu()

        # Use max of personal mean vs global as the method's accuracy
        test_acc = max(mean_personal, global_acc)
        if test_acc > best_acc: best_acc = test_acc
        history.append({'round': rnd, 'test_acc': float(test_acc),
                        'global_acc': float(global_acc),
                        'mean_personal_acc': float(mean_personal)})
        if rnd % 5 == 0 or rnd == 1:
            print(f'  R{rnd:2d}/{num_rounds} global={global_acc:.4f} personal_mean={mean_personal:.4f}')
    print(f'✓ DITTO done in {time.time()-t0:.0f}s — Best: {best_acc:.4f}')
    return history, best_acc, None


def run_fedrep(warmstart_sd, client_loaders, calib_loader, test_loader, num_rounds=NUM_ROUNDS):
    """FedRep: shared body (all except fc) + local head (fc). Only body aggregated."""
    global_model = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
    global_model.load_state_dict(copy.deepcopy(warmstart_sd))
    # Local heads per client
    local_heads = {}
    for cid in range(NUM_CLIENTS):
        local_heads[cid] = {
            'fc.weight': warmstart_sd['fc.weight'].clone(),
            'fc.bias': warmstart_sd['fc.bias'].clone(),
        }
    device = torch.device(DEVICE)
    history = []; best_acc = 0.0
    print(f'\n{"="*50}\nFEDREP — {num_rounds} rounds\n{"="*50}')
    t0 = time.time()
    for rnd in range(1, num_rounds+1):
        global_sd = copy.deepcopy(global_model.state_dict())
        body_weights = []
        for cid, loader in enumerate(client_loaders):
            local = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
            local.load_state_dict(global_sd)
            # Restore local head
            for k, v in local_heads[cid].items():
                local.state_dict()[k].copy_(v)
            local.to(device)
            crit = nn.CrossEntropyLoss()

            # Phase 1: Train head only (freeze body)
            for p in local.parameters(): p.requires_grad = False
            for p in local.fc.parameters(): p.requires_grad = True
            opt_head = torch.optim.SGD(local.fc.parameters(), lr=LR, momentum=MOMENTUM)
            local.train()
            for _ in range(FEDREP_HEAD_EPOCHS):
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    opt_head.zero_grad(); loss = crit(local(xb), yb)
                    loss.backward(); opt_head.step()

            # Phase 2: Train body only (freeze head)
            for p in local.parameters(): p.requires_grad = True
            for p in local.fc.parameters(): p.requires_grad = False
            body_params = [p for n, p in local.named_parameters() if 'fc' not in n]
            opt_body = torch.optim.SGD(body_params, lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
            for _ in range(FEDREP_BODY_EPOCHS):
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    opt_body.zero_grad(); loss = crit(local(xb), yb)
                    loss.backward(); opt_body.step()

            # Save local head
            local_heads[cid] = {k: local.state_dict()[k].cpu().clone()
                                for k in ['fc.weight', 'fc.bias']}
            # Collect body params for aggregation (exclude fc)
            n = len(loader.dataset)
            body_sd = {k: v.cpu() for k, v in local.state_dict().items() if 'fc' not in k}
            body_weights.append((n, body_sd))
            del local; torch.cuda.empty_cache()

        # Aggregate body only
        total = sum(n for n, _ in body_weights)
        new_body = OrderedDict({k: sum(n*sd[k].float() for n, sd in body_weights)/total
                                for k in body_weights[0][1]})
        gsd = global_model.state_dict()
        for k in new_body: gsd[k] = new_body[k].to(gsd[k].dtype)
        global_model.load_state_dict(gsd)

        # Evaluate: mean of personalized models (body + local head)
        accs = []
        for cid in range(NUM_CLIENTS):
            eval_model = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
            eval_sd = copy.deepcopy(global_model.state_dict())
            for k, v in local_heads[cid].items():
                eval_sd[k] = v
            eval_model.load_state_dict(eval_sd)
            eval_model.to(device)
            calibrate_bn(eval_model, client_loaders[cid], device, max_batches=10)
            accs.append(evaluate(eval_model, test_loader, device))
            del eval_model; torch.cuda.empty_cache()
        test_acc = float(np.mean(accs))
        if test_acc > best_acc: best_acc = test_acc
        history.append({'round': rnd, 'test_acc': test_acc, 'client_accs': accs})
        if rnd % 5 == 0 or rnd == 1:
            print(f'  R{rnd:2d}/{num_rounds} mean_personal={test_acc:.4f} worst={min(accs):.4f}')
    print(f'✓ FEDREP done in {time.time()-t0:.0f}s — Best: {best_acc:.4f}')
    return history, best_acc, None


def run_apfl(warmstart_sd, client_loaders, calib_loader, test_loader,
             num_rounds=NUM_ROUNDS, alpha_init=APFL_ALPHA_INIT):
    """APFL: adaptive mixing between global and local models per client."""
    global_model = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
    global_model.load_state_dict(copy.deepcopy(warmstart_sd))
    # Per-client: local model + mixing alpha
    local_models = [build_model(NUM_CLASSES, 3, **MODEL_KWARGS) for _ in range(NUM_CLIENTS)]
    for lm in local_models: lm.load_state_dict(copy.deepcopy(warmstart_sd))
    alphas = [alpha_init] * NUM_CLIENTS
    device = torch.device(DEVICE)
    history = []; best_acc = 0.0
    print(f'\n{"="*50}\nAPFL (α₀={alpha_init}) — {num_rounds} rounds\n{"="*50}')
    t0 = time.time()
    for rnd in range(1, num_rounds+1):
        global_sd = copy.deepcopy(global_model.state_dict())
        client_weights_g = []
        personal_accs = []
        for cid, loader in enumerate(client_loaders):
            alpha = alphas[cid]
            # Train global copy
            local_g = build_model(NUM_CLASSES, 3, **MODEL_KWARGS)
            local_g.load_state_dict(global_sd)
            _local_train(local_g, loader, device, LR, EPOCHS_PER_ROUND)
            n = len(loader.dataset)
            client_weights_g.append((n, {k:v.cpu() for k,v in local_g.state_dict().items()}))

            # Train local copy
            local_p = local_models[cid]
            _local_train(local_p, loader, device, LR, EPOCHS_PER_ROUND)

            # Mix: personalized = alpha * local + (1-alpha) * global
            mixed_sd = OrderedDict()
            for k in local_g.state_dict():
                mixed_sd[k] = alpha * local_p.state_dict()[k].cpu().float() + \
                              (1-alpha) * local_g.state_dict()[k].cpu().float()
            local_p.load_state_dict({k: v.to(local_p.state_dict()[k].dtype) for k, v in mixed_sd.items()})

            # Evaluate mixed model
            local_p.to(device)
            acc = evaluate(local_p, test_loader, device)
            personal_accs.append(acc)
            local_p.cpu()

            # Update alpha (simplified: increase if personal > global)
            global_model.to(device)
            g_acc = evaluate(local_g, test_loader, device)
            if acc > g_acc:
                alphas[cid] = min(alphas[cid] + 0.02, 1.0)
            else:
                alphas[cid] = max(alphas[cid] - 0.02, 0.0)
            global_model.cpu()
            del local_g; torch.cuda.empty_cache()

        # Aggregate global model
        new_sd = fedavg_aggregate(global_sd, client_weights_g)
        gsd = global_model.state_dict()
        for k in new_sd: gsd[k] = new_sd[k].to(gsd[k].dtype)
        global_model.load_state_dict(gsd)

        test_acc = float(np.mean(personal_accs))
        if test_acc > best_acc: best_acc = test_acc
        history.append({'round': rnd, 'test_acc': test_acc, 'alphas': list(alphas)})
        if rnd % 5 == 0 or rnd == 1:
            print(f'  R{rnd:2d}/{num_rounds} personal_mean={test_acc:.4f} α={[f"{a:.2f}" for a in alphas]}')
    print(f'✓ APFL done in {time.time()-t0:.0f}s — Best: {best_acc:.4f}')
    return history, best_acc, None

print('All FL methods defined ✓')

All FL methods defined ✓


## 5. Run All FL Methods (Quick: 10 Rounds Each)

In [8]:
fedavg_h, fedavg_best, _ = run_fedavg(warmstart_sd, client_loaders, calib_loader, test_loader)


FEDAVG — 10 rounds
  R 1/10 test=0.5507
  R 5/10 test=0.6726
  R10/10 test=0.6570
✓ FEDAVG done in 843s — Best: 0.7083


In [9]:
fedbn_h, fedbn_best, _ = run_fedbn(warmstart_sd, client_loaders, calib_loader, test_loader)


FEDBN — 10 rounds
  R 1/10 test=0.5355
  R 5/10 test=0.6715
  R10/10 test=0.7126
✓ FEDBN done in 841s — Best: 0.7188


In [10]:
fedprox_h, fedprox_best, _ = run_fedprox(warmstart_sd, client_loaders, calib_loader, test_loader, mu=0.01)


FEDPROX — 10 rounds
  R 1/10 test=0.5355
  R 5/10 test=0.6493
  R10/10 test=0.7341
✓ FEDPROX done in 883s — Best: 0.7341


In [11]:
ditto_h, ditto_best, _ = run_ditto(warmstart_sd, client_loaders, calib_loader, test_loader, lam=0.1)


DITTO (λ=0.1) — 10 rounds
  R 1/10 global=0.5558 personal_mean=0.4389
  R 5/10 global=0.6719 personal_mean=0.5082
  R10/10 global=0.7050 personal_mean=0.5756
✓ DITTO done in 1882s — Best: 0.7134


In [12]:
fedrep_h, fedrep_best, _ = run_fedrep(warmstart_sd, client_loaders, calib_loader, test_loader)


FEDREP — 10 rounds
  R 1/10 mean_personal=0.5583 worst=0.5318
  R 5/10 mean_personal=0.6091 worst=0.5467
  R10/10 mean_personal=0.6405 worst=0.5609
✓ FEDREP done in 1348s — Best: 0.6405


## 6. FL Methods Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 10})

all_methods = [
    ('FedAvg',    fedavg_h,  fedavg_best,  '#3498db'),
    ('FedBN',     fedbn_h,   fedbn_best,   '#e74c3c'),
    ('FedProx',   fedprox_h, fedprox_best, '#f39c12'),
    ('Ditto',     ditto_h,   ditto_best,   '#2ecc71'),
    ('FedRep',    fedrep_h,  fedrep_best,  '#9b59b6'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Bar chart
names = [m[0] for m in all_methods]
bests = [m[2] for m in all_methods]
colors = [m[3] for m in all_methods]
bars = axes[0].bar(names, bests, color=colors, edgecolor='k', linewidth=0.5)
axes[0].set_ylabel('Best Test Accuracy')
axes[0].set_title(f'Best Accuracy ({NUM_ROUNDS} Rounds)')
axes[0].set_ylim(min(bests)-0.05, max(bests)+0.05)
for bar, acc in zip(bars, bests):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{acc:.3f}', ha='center', fontweight='bold', fontsize=9)

# Panel 2: Learning curves
for name, hist, _, color in all_methods:
    rounds = [h['round'] for h in hist]
    accs = [h['test_acc'] for h in hist]
    axes[1].plot(rounds, accs, '-o', color=color, label=name, linewidth=1.5, markersize=3)
axes[1].set_xlabel('FL Round'); axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Learning Curves'); axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'FL Methods Comparison — {NUM_ROUNDS} Rounds Quick Scan', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/methods_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"="*55}')
print(f'{"Method":<12s} {"Best Acc":>10s} {"Final Acc":>10s} {"Δ vs FedAvg":>12s}')
print(f'{"-"*55}')
for name, hist, best, _ in all_methods:
    final = hist[-1]['test_acc']
    delta = best - fedavg_best
    print(f'{name:<12s} {best:>10.4f} {final:>10.4f} {delta:>+12.4f}')
print(f'{"="*55}')